# 实验结果汇总表
本 Notebook 自动读取 `ranking` 和 `aft` 文件夹下的实验结果，并一键生成按不同对照组分类的 Markdown 表格。

- **Ranking Base**: m=10, n=100, pc=0.3
- **AFT Base**: m=10, n=100, pc=0.3, cens=0.2

运行下方代码块即可生成所有表格结果。

In [4]:
import os
import json
import numpy as np
import glob
from collections import defaultdict
from IPython.display import display, Markdown

def parse_filename(f):
    basename = os.path.basename(f)
    parts = basename.replace('.json', '').split('_')
    parts[-1] = parts[-1].split(' ')[0] # handle ' (1)'
    
    info = {'noise': parts[0]}
    for p in parts[1:]:
        if p.startswith('m'): info['m'] = p[1:]
        elif p.startswith('n') and not p.startswith('normal') and not p.startswith('noise'): info['n'] = p[1:]
        elif p.startswith('pc'): info['pc'] = p[2] + '.' + p[3:] if len(p)>3 else p[2:]
        elif p.startswith('cens'): info['cens'] = p[4] + '.' + p[5:] if len(p)>5 else p[4:]
    return info

def get_setting_name(info, task):
    m = info.get('m')
    n = info.get('n')
    pc = info.get('pc')
    cens = info.get('cens', '0.2') # default to 0.2 if not found
    
    if task == 'ranking':
        if m == '10' and n == '100' and pc == '0.3': return 'Base'
        if m == '20' and n == '100' and pc == '0.3': return 'm=20'
        if m == '10' and n == '200' and pc == '0.3': return 'n=200'
        if m == '10' and n == '100' and pc == '0.5': return 'pc=0.5'
        return 'Base'
    else:
        # For AFT, Base is m=10, n=100, pc=0.3, cens=0.2
        if m == '10' and n == '100' and pc == '0.3' and (cens == '0.2' or cens == '02'): return 'Base'
        if m == '20' and n == '100' and pc == '0.3' and (cens == '0.2' or cens == '02'): return 'm=20'
        if m == '10' and n == '200' and pc == '0.3' and (cens == '0.2' or cens == '02'): return 'n=200'
        if m == '10' and n == '100' and pc == '0.5' and (cens == '0.2' or cens == '02'): return 'pc=0.5'
        if m == '10' and n == '100' and pc == '0.3' and (cens == '0.3' or cens == '03'): return 'cens=0.3'
        if m == '10' and n == '100' and pc == '0.3' and (cens == '0.4' or cens == '04'): return 'cens=0.4'
        return 'Base'

def generate_table(task, folder):
    if not os.path.exists(folder): return "Folder not found."
    files = glob.glob(os.path.join(folder, '*.json'))
    data_dict = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))
    methods = ['Global', 'Local', 'Avg', 'D-ProxGD', 'U-ADMM']
    
    for f in files:
        info = parse_filename(f)
        noise = info['noise']
        setting = get_setting_name(info, task)
        
        with open(f, 'r', encoding='utf-8') as file:
            try:
                data = json.load(file)
                for r in data.get('results', []):
                    for m in methods:
                        if m in r and 'RMSE' in r[m]:
                            data_dict[noise][setting][m].append(r[m]['RMSE'])
            except:
                pass
                
    if task == 'ranking':
        cols = ['Base', 'm=20', 'n=200', 'pc=0.5']
    else:
        cols = ['Base', 'm=20', 'n=200', 'pc=0.5', 'cens=0.3', 'cens=0.4']
        
    md_output = ""
    for noise in sorted(data_dict.keys()):
        md_output += f"### Task: {task.capitalize()} | Noise: {noise}\n\n"
        header = "| Method | " + " | ".join(cols) + " |"
        separator = "|---" + "|---" * len(cols) + "|"
        md_output += header + "\n" + separator + "\n"
        
        for m in methods:
            row = f"| {m} | "
            vals = []
            for c in cols:
                rmses = data_dict[noise][c][m]
                if len(rmses) > 0:
                    vals.append(f"{np.mean(rmses):.4f}")
                else:
                    vals.append("-")
            row += " | ".join(vals) + " |"
            md_output += row + "\n"
        md_output += "\n"
    return md_output

md_ranking = generate_table('ranking', 'ranking')
md_aft = generate_table('aft', 'aft')

display(Markdown("# Ranking 实验汇总"))
display(Markdown(md_ranking))
display(Markdown("# AFT 实验汇总"))
display(Markdown(md_aft))


# Ranking 实验汇总

### Task: Ranking | Noise: exp

| Method | Base | m=20 | n=200 | pc=0.5 |
|---|---|---|---|---|
| Global | 0.0883 | 0.0860 | 0.0835 | 0.0883 |
| Local | 0.3687 | 0.3688 | 0.2698 | 0.3687 |
| Avg | 0.1787 | 0.1586 | 0.1475 | 0.1787 |
| D-ProxGD | 0.0995 | 0.1053 | 0.1008 | 0.1009 |
| U-ADMM | 0.0923 | 0.0871 | 0.0849 | 0.0896 |

### Task: Ranking | Noise: normal

| Method | Base | m=20 | n=200 | pc=0.5 |
|---|---|---|---|---|
| Global | 0.0857 | 0.0830 | 0.0804 | 0.0857 |
| Local | 0.3681 | 0.3720 | 0.2706 | 0.3681 |
| Avg | 0.1750 | 0.1598 | 0.1508 | 0.1750 |
| D-ProxGD | 0.0953 | 0.1033 | 0.1018 | 0.0962 |
| U-ADMM | 0.0888 | 0.0860 | 0.0842 | 0.0862 |

### Task: Ranking | Noise: t1

| Method | Base | m=20 | n=200 | pc=0.5 |
|---|---|---|---|---|
| Global | 0.0871 | 0.0760 | 0.0750 | 0.1102 |
| Local | 0.5619 | 0.6014 | 0.4827 | 0.6018 |
| Avg | 0.2100 | 0.1955 | 0.1891 | 0.2454 |
| D-ProxGD | 0.0899 | 0.0790 | 0.0813 | 0.1094 |
| U-ADMM | 0.0899 | 0.0749 | 0.0776 | 0.1118 |



# AFT 实验汇总

### Task: Aft | Noise: exp

| Method | Base | m=20 | n=200 | pc=0.5 | cens=0.3 | cens=0.4 |
|---|---|---|---|---|---|---|
| Global | 0.0848 | 0.0615 | 0.0614 | 0.0848 | 0.0881 | 0.0918 |
| Local | 0.5677 | 0.5628 | 0.3385 | 0.5677 | 0.5979 | 0.6474 |
| Avg | 0.1778 | 0.1397 | 0.1076 | 0.1778 | 0.1898 | 0.2097 |
| D-ProxGD | 0.0942 | 0.0714 | 0.0681 | 0.0922 | 0.1101 | 0.1166 |
| U-ADMM | 0.0885 | 0.0621 | 0.0635 | 0.0911 | 0.1000 | 0.1008 |

### Task: Aft | Noise: gumbel

| Method | Base | m=20 | n=200 | pc=0.5 | cens=0.3 | cens=0.4 |
|---|---|---|---|---|---|---|
| Global | 0.1260 | 0.0980 | 0.0911 | 0.1239 | 0.1326 | 0.1404 |
| Local | 0.8136 | 0.8016 | 0.5108 | 0.8136 | 0.8516 | 0.9065 |
| Avg | 0.2504 | 0.1849 | 0.1654 | 0.2504 | 0.2616 | 0.2839 |
| D-ProxGD | 0.1544 | 0.1210 | 0.1118 | 0.1421 | 0.1590 | 0.1667 |
| U-ADMM | 0.1329 | 0.1064 | 0.0953 | 0.1296 | 0.1405 | 0.1419 |

### Task: Aft | Noise: normal

| Method | Base | m=20 | n=200 | pc=0.5 | cens=0.3 | cens=0.4 |
|---|---|---|---|---|---|---|
| Global | 0.1099 | 0.0843 | 0.0807 | 0.1126 | 0.1165 | 0.1200 |
| Local | 0.7113 | 0.7174 | 0.4718 | 0.7113 | 0.7644 | 0.8363 |
| Avg | 0.2275 | 0.1678 | 0.1486 | 0.2275 | 0.2475 | 0.2765 |
| D-ProxGD | 0.1316 | 0.1044 | 0.1019 | 0.1421 | 0.1501 | 0.1705 |
| U-ADMM | 0.1122 | 0.0854 | 0.0835 | 0.1159 | 0.1201 | 0.1251 |

### Task: Aft | Noise: t1

| Method | Base | m=20 | n=200 | pc=0.5 | cens=0.3 | cens=0.4 |
|---|---|---|---|---|---|---|
| Global | 0.2134 | 0.1654 | 0.1597 | 0.2134 | 0.2415 | 0.2628 |
| Local | 1.6122 | 1.6209 | 0.9398 | 1.6122 | 1.8292 | 2.2079 |
| Avg | 0.5105 | 0.3724 | 0.2833 | 0.5105 | 0.6036 | 0.7553 |
| D-ProxGD | 0.2609 | 0.1950 | 0.1820 | 0.2568 | 0.3177 | 0.4297 |
| U-ADMM | 0.2292 | 0.1773 | 0.1594 | 0.2203 | 0.2629 | 0.2833 |

